# Run LLM Judge Experiments

This notebook only orchestrates the experiment.  
All metrics and logging are imported from `evaluation_pipeline`.

### 1. IMPORTS

In [ ]:
import pandas as pd
import os
import random
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from tqdm import tqdm
from datetime import datetime
from pathlib import Path

# Custom modules
from evaluation_pipeline.prompts import load_prompt_template, build_experiment_prompt
from evaluation_pipeline.dataset import load_dataset_from_file
from evaluation_pipeline.judge import judge_response
from evaluation_pipeline.runner import run_judge_experiment
from evaluation_pipeline.metrics import (
    compute_classification_metrics,
    compute_output_quality,
    compute_second_level_metrics,
    compute_difference_metrics,
    compute_dynamic_prompt_metrics,
    compute_flash_metric,
    build_summary_metrics,
    compare_methods,
    TASK_LABELS,
)
from evaluation_pipeline.wandb_logger import log_experiment
from sklearn.metrics import ConfusionMatrixDisplay

### 2. CONFIG

In [ ]:
RUN_ID = datetime.now().strftime("r_%m%d_%H%M")

JUDGE_MODEL = "llama3.2:3b"  # Changed from llama3:8b to avoid memory error
JUDGE_METHOD = "second_level"   # baseline, second_level, dynamic (dynamic not implemented yet)

BASELINE_PROMPT_FILE = "../prompts/baseline_truthfulness_v1.txt"
SECOND_LEVEL_PROMPT_FILE = "../prompts/second_level_truthfulness_v2.txt"
# DYNAMIC_PROMPT_FILES = ["../prompts/dynamic_v1.txt", "../prompts/dynamic_v2.txt"]  # optional, not used yet

DATASET_FILE = "../datasets/prepared/liar_40.json"
TASK_TYPE = "truthfulness"

WANDB_PROJECT_NAME = "judge_evaluation"
WANDB_ENABLED = True

In [ ]:
VALID_METHODS = {"baseline", "dynamic", "second_level"}
VALID_TASK_TYPES = {"truthfulness", "safety", "liar"}

if JUDGE_METHOD not in VALID_METHODS:
    raise ValueError(f"Invalid JUDGE_METHOD: {JUDGE_METHOD}")
if TASK_TYPE not in VALID_TASK_TYPES:
    raise ValueError(f"Invalid TASK_TYPE: {TASK_TYPE}")

REQUIRED_PROMPTS_BY_METHOD = {
    "baseline": ["baseline"],
    "dynamic": ["baseline", "dynamic"],
    "second_level": ["baseline", "second_level"],
}
required_prompts = REQUIRED_PROMPTS_BY_METHOD[JUDGE_METHOD]

print("Config validation passed.")
print(f"JUDGE_METHOD: {JUDGE_METHOD}")
print(f"TASK_TYPE: {TASK_TYPE}")
print(f"Required prompts: {required_prompts}")

### 3. Load Dataset

In [ ]:
dataset = load_dataset_from_file(DATASET_FILE)
seed = 42
random.Random(seed).shuffle(dataset)
print(f"Loaded dataset contains {len(dataset)} entries.")
print(json.dumps(dataset[0], indent=2))

### 4. Load Prompt Templates

In [ ]:
templates = {}
if "baseline" in required_prompts:
    templates["baseline"] = load_prompt_template(BASELINE_PROMPT_FILE)
if "second_level" in required_prompts:
    templates["second_level"] = load_prompt_template(SECOND_LEVEL_PROMPT_FILE)
# if "dynamic" in required_prompts:
#     templates["dynamic"] = load_prompt_template(DYNAMIC_PROMPT_FILES[0])  # placeholder
print("Loaded prompt templates:", list(templates.keys()))

### 5. Run Judge Experiment

In [ ]:
results = run_judge_experiment(
    dataset=dataset,
    run_id=RUN_ID,
    model=JUDGE_MODEL,
    method=JUDGE_METHOD,
    templates=templates,
    dataset_file=DATASET_FILE,
)

### 6. Save Predictions

In [ ]:
df_results = pd.DataFrame(results)
os.makedirs("../results", exist_ok=True)
RESULTS_FILE = f"../results/{RUN_ID}_predictions.csv"
df_results.to_csv(RESULTS_FILE, index=False)
print(f"Saved predictions to: {RESULTS_FILE}")

### 7. Compute Metrics (using imported functions)

In [ ]:
labels = TASK_LABELS[TASK_TYPE]

# --- Output quality ---
quality = compute_output_quality(results, labels)
print("Output quality:", quality)

# --- Classification metrics ---
metrics = compute_classification_metrics(results, labels)
print("Classification metrics:", {k: v for k, v in metrics.items() if k not in ["classification_report", "confusion_matrix"]})

# --- Second-level metrics (if applicable) ---
if JUDGE_METHOD == "second_level":
    sl_metrics = compute_second_level_metrics(results)
    print("Second-level metrics:", sl_metrics)

# --- Flash metric (response length) ---
flash = compute_flash_metric(results)
print("Flash metric:", flash)

# --- Summary (single row) ---
summary = build_summary_metrics(
    results=results,
    labels=labels,
    run_id=RUN_ID,
    method=JUDGE_METHOD,
    model=JUDGE_MODEL,
    task_type=TASK_TYPE,
    dataset_file=DATASET_FILE,
    baseline_prompt_file=BASELINE_PROMPT_FILE,
    second_level_prompt_file=SECOND_LEVEL_PROMPT_FILE if JUDGE_METHOD == "second_level" else "",
)
df_summary = pd.DataFrame([summary])
print("Summary:", summary)

### 8. Confusion Matrix Plot

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = metrics["confusion_matrix"]
fig, ax = plt.subplots(figsize=(7, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap=plt.cm.Blues, ax=ax, values_format="d")
for text in disp.text_.flatten():
    text.set_fontsize(24)
ax.tick_params(axis="both", labelsize=16)
ax.set_xlabel("Predicted label", fontsize=16)
ax.set_ylabel("True label", fontsize=16)
ax.set_title(f"{TASK_TYPE} Confusion Matrix:\n{RUN_ID}-{JUDGE_MODEL} {JUDGE_METHOD}", fontsize=18, pad=15)
plt.tight_layout()
plt.show()

### 9. Save Excel Report

In [ ]:
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

safe_model = JUDGE_MODEL.replace(":", "_").replace("/", "_")
base_name = f"{RUN_ID}_{JUDGE_METHOD}_{safe_model}_{TASK_TYPE}"
excel_path = RESULTS_DIR / f"{base_name}.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_results.to_excel(writer, sheet_name="raw_results", index=False)
    df_summary.to_excel(writer, sheet_name="summary_metrics", index=False)

    if JUDGE_METHOD == "second_level":
        # second-level case analysis (re‑use notebook logic or compute directly)
        df_sl = df_results.copy()
        first_correct = df_sl["first_level_label"] == df_sl["true_label"]
        final_correct = df_sl["predicted_label"] == df_sl["true_label"]
        corrected = (~first_correct) & final_correct
        degraded = first_correct & (~final_correct)
        unchanged_correct = first_correct & final_correct
        unchanged_wrong = (~first_correct) & (~final_correct)
        df_sl["case_type"] = "unknown"
        df_sl.loc[corrected, "case_type"] = "corrected"
        df_sl.loc[degraded, "case_type"] = "degraded"
        df_sl.loc[unchanged_correct, "case_type"] = "unchanged_correct"
        df_sl.loc[unchanged_wrong, "case_type"] = "unchanged_wrong"
        df_second_level_cases = df_sl[[
            "id", "true_label", "first_level_label", "second_level_verdict",
            "predicted_label", "case_type", "first_level_explanation", "second_level_explanation"
        ]].copy()
        df_second_level_metrics = pd.DataFrame([sl_metrics])
        df_second_level_metrics.to_excel(writer, sheet_name="second_level_metrics", index=False)
        df_second_level_cases.to_excel(writer, sheet_name="second_level_cases", index=False)

print("Saved experiment report:", excel_path)

### 10. Log to Weights & Biases

In [ ]:
config = {
    "run_id": RUN_ID,
    "method": JUDGE_METHOD,
    "model": JUDGE_MODEL,
    "task_type": TASK_TYPE,
    "dataset_file": DATASET_FILE,
    "baseline_prompt_file": BASELINE_PROMPT_FILE,
    "second_level_prompt_file": SECOND_LEVEL_PROMPT_FILE if JUDGE_METHOD == "second_level" else "",
}

log_experiment(
    project_name=WANDB_PROJECT_NAME,
    run_name=f"{JUDGE_MODEL}_{JUDGE_METHOD}_{TASK_TYPE}_{len(dataset)}_{RUN_ID}",
    config=config,
    df_results=df_results,
    df_summary_metrics=df_summary,
    df_second_level_metrics=df_second_level_metrics if JUDGE_METHOD == "second_level" else None,
    df_second_level_cases=df_second_level_cases if JUDGE_METHOD == "second_level" else None,
    confusion_figure=fig,
    enabled=WANDB_ENABLED,
)